In [0]:
%pip install azure-eventhub

In [0]:
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("evh_cs_key", "")     
dbutils.widgets.text("eventhub_name", "")
dbutils.widgets.text("num_events", "100")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

secret_scope  = dbutils.widgets.get("secret_scope")
evh_cs_key    = dbutils.widgets.get("evh_cs_key")
eventhub_name = dbutils.widgets.get("eventhub_name")
num_events    = int(dbutils.widgets.get("num_events"))
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")

connection_str = dbutils.secrets.get(scope=secret_scope, key=evh_cs_key)

In [0]:
eh_namespace = "evhpl24databricks02"
bootstrap_servers = f"{eh_namespace}.servicebus.windows.net:9093"

eh_sasl = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{connection_str}";'
)

In [0]:
eh_base_path       = f"/Volumes/{catalog}/{schema}/streaming_landing/eventhub"
eh_checkpoint_peek = f"{eh_base_path}/_checkpoint_peek"
eh_checkpoint      = f"{eh_base_path}/_checkpoint"

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json, random
from datetime import datetime

producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_str, eventhub_name=eventhub_name
)
with producer:
    batch = producer.create_batch()
    for i in range(num_events):
        tx = {
            "transaction_id": f"evh_{i}",
            "amount": random.randint(100, 1000),
            "currency": random.choice(["USD", "EUR", "PLN"]),
            "category": random.choice(["food", "tech", "travel"]),
            "event_time": datetime.now().isoformat(),
        }
        batch.add(EventData(json.dumps(tx)))
    producer.send_batch(batch)
print(f"Sen {num_events} transactions")

In [0]:
df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", eventhub_name)                   
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", eh_sasl)
    .option("startingOffsets", "earliest")             
    .load()
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

transaction_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("amount", IntegerType()),
    StructField("currency", StringType()),
    StructField("category", StringType()),
    StructField("event_time", StringType()),
])

df_parsed = (
    df_kafka
    .withColumn("value_str", F.col("value").cast("string"))
    .withColumn("data", F.from_json(F.col("value_str"), transaction_schema))
    .select(
        "data.*",                                
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
    )
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
    .withColumn("source", F.lit("eventhub"))
)

In [0]:
eh_bronze_table = f"{catalog}.{schema}.transactions_eventhub"

query_eh = (
    df_parsed.writeStream
    .format("delta")
    .option("checkpointLocation", eh_checkpoint)  
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(eh_bronze_table)
)

query_eh.awaitTermination()
print("Stream Event Hub ended.")

In [0]:
spark.catalog.refreshTable(eh_bronze_table)
display(spark.sql(f"SELECT * FROM {eh_bronze_table} LIMIT 10"))